In [3]:
import PyPDF2

file = open("data/books/book.pdf", "rb")
reader = PyPDF2.PdfReader(file)

for i in range(len(reader.pages)):
    page = reader.pages[i]

    writer = PyPDF2.PdfWriter()
    writer.add_page(page)

    output_path = f"data/books_split/part_{i+1}.pdf"

    with open(output_path, "wb") as output_file:
        writer.write(output_file)

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

endpoint = os.getenv("AZURE_ENDPOINT")
key = os.getenv("AZURE_KEY")

print(endpoint)
print(key[:5])

https://ocr-norhan.cognitiveservices.azure.com/
9PO3W


In [13]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

endpoint = os.getenv("AZURE_ENDPOINT")
key = os.getenv("AZURE_KEY")

client = DocumentAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

In [38]:
import os
import json
import time

input_folder = "data/books_split"
output_file = "data/books_proc/book1_ocr.json"

files = sorted(
    os.listdir(input_folder),
    key=lambda x: int(x.split("_")[1].split(".")[0])
)[:20]

ocr_results = []

for file_name in files:
    if not file_name.endswith(".pdf"):
        continue

    file_path = os.path.join(input_folder, file_name)
    print("processing:", file_name)

    try:
        with open(file_path, "rb") as f:
            poller = client.begin_analyze_document(
                "prebuilt-read",
                document=f
            )
            result = poller.result()

        page_text = ""

        for page in result.pages:
            for line in page.lines:
                page_text += line.content + "\n"

        ocr_results.append({
            "file_name": file_name,
            "text": page_text.strip()
        })

    except Exception as e:
        print("error in", file_name, ":", e)

    time.sleep(1)

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(ocr_results, f, ensure_ascii=False, indent=2)

processing: part_1.pdf
processing: part_2.pdf
processing: part_3.pdf
processing: part_4.pdf
processing: part_5.pdf
processing: part_6.pdf
processing: part_7.pdf
processing: part_8.pdf
processing: part_9.pdf
processing: part_10.pdf
processing: part_11.pdf
processing: part_12.pdf
processing: part_13.pdf
processing: part_14.pdf
processing: part_15.pdf
processing: part_16.pdf
processing: part_17.pdf
processing: part_18.pdf
processing: part_19.pdf
processing: part_20.pdf


In [39]:
import json

json_path = "data/books_proc/book1_ocr.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

all_sentences = []

for item in data:
    text = item["text"]
    sentences = text.split("\n")  
    all_sentences.extend(sentences)

all_sentences = [s.strip() for s in all_sentences if s.strip() != ""]

print("Total sentences:", len(all_sentences))
print(all_sentences[:10])


Total sentences: 349
['إصدفل', '1(3) كيف تكسب', 'وتؤتر في الناس', 'ديل كارنيجي', 'اهلية', 'كيف تكسب', 'الأصدقاء', 'وتؤتر في الناس', 'كمية', 'الأهلية للنشر والتوزيع']


In [40]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1")

embeddings = model.encode(all_sentences)

print("Shape:", embeddings.shape)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7817.64it/s]


Shape: (349, 768)


In [42]:
import faiss
import numpy as np
import pickle

In [43]:
dimension = embeddings.shape[1]  

index = faiss.IndexFlatL2(dimension)

In [44]:
index.add(np.array(embeddings))

In [45]:
faiss.write_index(index, "data/vecs/vector_index.faiss")

with open("data/vecs/documents.pkl", "wb") as f:
    pickle.dump(all_sentences, f)

In [46]:
index = faiss.read_index("data/vecs/vector_index.faiss")

with open("data/vecs/documents.pkl", "rb") as f:
    docs = pickle.load(f)

In [47]:
query = "كيف اكسب الاصدقاء"

query_vector = model.encode([query])

In [48]:
k = 3

distances, indices = index.search(np.array(query_vector), k)

In [49]:
print("Query:", query)

for i, idx in enumerate(indices[0]):
    print(f"{i+1}. {docs[idx]} (distance={distances[0][i]:.4f})")

Query: كيف اكسب الاصدقاء
1. كيف تكسب الأصدقاء (distance=54.2102)
2. - كيف تكسب الأصدقاء وتؤثر في الناس (distance=138.1578)
3. - كيف تكسب الأصدقاء وتؤثر في الناس (distance=138.1578)


In [65]:
import sys
!"{sys.executable}" -m pip install faiss-cpu


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
